# MediShield — Advanced Offline ML Training Pipeline (Google Colab CPU)

This notebook trains lightweight, **100% offline, LLM-free** models for **MediShield (Track 3 — Secret Wards)**:
1. **50,000** annotated synthetic clinical records (20k English, 15k Hindi, 15k Hinglish) for PII context detection.
2. **50,000** labeled cross-hospital record pairs (matches, hard-negatives, missing fields, multilingual/spelling variants) for Record Linkage.
3. Grouped **70% Train / 15% Validation / 15% Test** splits to prevent data leakage.
4. Exports `pii_model.joblib`, `linkage_model.joblib`, `model_metadata.json`, and `evaluation_report.json`.

### Cell 1 — Install Required Libraries
Installs `scikit-learn`, `rapidfuzz`, `pandas`, `numpy`, `joblib`, and `matplotlib` on Google Colab CPU runtime (no GPU needed).

In [ ]:
!pip -q install scikit-learn==1.6.1 rapidfuzz==3.11.0 numpy==2.2.1 pandas==2.2.3 joblib>=1.4.0 matplotlib>=3.9.0

### Cell 2 — Clone Repo or Upload `medishield/` Scripts
Clones your `M20A03/Yugaantar` repository so Colab can use the exact reproducible `generate_datasets.py` and `train_and_evaluate.py` scripts.

In [ ]:
import os
if not os.path.exists('Yugaantar'):
    !git clone https://github.com/M20A03/Yugaantar.git
%cd Yugaantar/medishield

### Cell 3 — Dataset Generation, Validation & Exploratory Data Analysis (EDA)
Generates 50,000 PII records + 50,000 linkage pairs, verifies exact counts and 70/15/15 grouped splits, and prints class/language distributions.

In [ ]:
import sys
from collections import Counter
sys.path.insert(0, 'training')
from generate_datasets import generate_pii_dataset, generate_linkage_dataset, group_split

pii_data = generate_pii_dataset(20000, 15000, 15000, seed=2026)
link_data = generate_linkage_dataset(seed=2026)
p_tr, p_va, p_te = group_split(pii_data, lambda x: x['group_id'])
l_tr, l_va, l_te = group_split(link_data, lambda x: x['group_id'])

print('PII Total:', len(pii_data), '| Languages:', Counter(r['language'] for r in pii_data))
print('PII Splits (Train/Val/Test):', len(p_tr), len(p_va), len(p_te))
print('Linkage Total:', len(link_data), '| Categories:', Counter(r['category'] for r in link_data))
print('Linkage Splits (Train/Val/Test):', len(l_tr), len(l_va), len(l_te))

### Cell 4 — Train & Compare PII Models + Record Linkage Models + Learning Curves
Runs `training/train_and_evaluate.py` which:
- Compares Word TF-IDF + LR vs. Character (2-4) TF-IDF + LR
- Compares Logistic Regression vs. Random Forest vs. HistGradientBoosting for Record Linkage
- Calibrates thresholds on the validation set
- Computes learning curves at 1k, 10k, 25k, and 50k samples
- Exports `pii_model.joblib`, `linkage_model.joblib`, `model_metadata.json`, and `evaluation_report.json`.

In [ ]:
from train_and_evaluate import train_and_evaluate
report = train_and_evaluate()

### Cell 5 — Evaluation Charts (Learning Curve & Model Comparison)
Visualizes how F1 score changes as training size grows from 1,000 to 35,000+ training pairs, and shows the confusion matrix.

In [ ]:
import matplotlib.pyplot as plt

lc = report['learning_curve']
sizes = [x['train_samples'] for x in lc]
f1s = [x.get('val_f1', x.get('test_f1', 0.0)) for x in lc]

plt.figure(figsize=(7, 4))
plt.plot(sizes, f1s, marker='o', linewidth=2)
plt.title('Record Linkage Learning Curve (Stratified Validation F1)')
plt.xlabel('Training Pairs')
plt.ylabel('Validation F1 Score')
plt.grid(True, alpha=0.3)
plt.show()

print('Confusion Matrix [ [TN, FP], [FN, TP] ]:', report['linkage_test_metrics']['confusion_matrix'])

### Cell 6 — Download Trained Artifacts for Offline Backend
Downloads the exported `.joblib` models and JSON reports from Google Colab to your local machine.

In [ ]:
try:
    from google.colab import files
    for fname in [
        'backend/artifacts/pii_model.joblib',
        'backend/artifacts/linkage_model.joblib',
        'backend/artifacts/model_metadata.json',
        'backend/artifacts/evaluation_report.json',
    ]:
        files.download(fname)
except ImportError:
    print('Not running in Colab; artifacts are saved locally in backend/artifacts/')